# 中国年报 PDF 解析

用 pdfplumber 提取 PDF 文字，正则切分章节（第X节 / 第X章），保存为 JSON。

In [1]:
import os, re, json, time, warnings
from datetime import datetime
import pdfplumber

warnings.filterwarnings("ignore", message=".*CropBox.*")

# ---- 路径配置 ----
PDF_DIR = os.path.join("..", "..", "01_CN_CNINF_report", "\u5e74\u62a5\u6587\u4ef6", "2024", "pdf\u5e74\u62a5")
OUT_DIR = os.path.join("parsed_filings")
os.makedirs(OUT_DIR, exist_ok=True)

print("PDF \u76ee\u5f55:", os.path.abspath(PDF_DIR))
print("\u8f93\u51fa\u76ee\u5f55:", os.path.abspath(OUT_DIR))

pdf_files = sorted([f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")])
print(f"\u5171\u627e\u5230 {len(pdf_files)} \u4e2a PDF")

PDF 目录: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/01_CN_CNINF_report/年报文件/2024/pdf年报
输出目录: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/03_Data_Processor/CN/parsed_filings
共找到 4790 个 PDF


## Step 1 & 2: 提取全文 + meta 信息

In [2]:
def extract_text_from_pdf(pdf_path):
    """\u7528 pdfplumber \u9010\u9875\u63d0\u53d6\u6587\u5b57\uff0c\u8fd4\u56de (\u5168\u6587, \u9875\u6570)\u3002"""
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        for page in pdf.pages:
            text = page.extract_text() or ""
            pages_text.append(text)
    full_text = "\n".join(pages_text)
    return full_text, total_pages


def parse_filename(filename):
    """\u4ece\u6587\u4ef6\u540d '000001_\u5e73\u5b89\u94f6\u884c_2024.pdf' \u89e3\u6790 stock_code, company_name, year\u3002"""
    name = filename.replace(".pdf", "")
    parts = name.split("_", 2)
    if len(parts) == 3:
        return parts[0], parts[1], parts[2]
    return name, "", ""


# \u5feb\u901f\u6d4b\u8bd5\u4e00\u4efd
test_file = "000001_\u5e73\u5b89\u94f6\u884c_2024.pdf"
test_path = os.path.join(PDF_DIR, test_file)

t0 = time.time()
text, pages = extract_text_from_pdf(test_path)
elapsed = time.time() - t0

code, name, year = parse_filename(test_file)
print(f"\u6587\u4ef6: {test_file}")
print(f"\u80a1\u7968\u4ee3\u7801: {code}, \u516c\u53f8: {name}, \u5e74\u4efd: {year}")
print(f"\u9875\u6570: {pages}, \u603b\u5b57\u6570: {len(text)}, \u8017\u65f6: {elapsed:.1f}s")
print(f"\u524d200\u5b57: {text[:200]}")

KeyboardInterrupt: 

## Step 3: 正则切分章节

In [ ]:
SECTION_PATTERN = re.compile(
    r"^\s*(\u7b2c[\u4e00\u4e8c\u4e09\u56db\u4e94\u516d\u4e03\u516b\u4e5d\u5341\u767e]+[\u8282\u7ae0]\s*.+?)\s*$",
    re.MULTILINE,
)

def clean_title(title):
    title = re.sub(r"[.\u3000\u00b7\u2026\s]+\d*\s*$", "", title)
    title = re.sub(r"\s{2,}", " ", title)
    return title.strip()


def clean_section_text(text):
    """\u53bb\u6389\u6240\u6709 PDF \u6362\u884c\uff0c\u62fc\u6210\u8fde\u7eed\u6587\u672c\u3002"""
    return text.replace("\n", "")


def split_sections(full_text):
    """
    \u7528\u6b63\u5219\u627e\u5230\u6240\u6709 '\u7b2cX\u8282/\u7b2cX\u7ae0' \u6807\u9898\uff0c\u6309\u4f4d\u7f6e\u5207\u5206\u6587\u672c\u3002
    \u8fd4\u56de dict: {\u7ae0\u8282\u6807\u9898: \u7ae0\u8282\u6b63\u6587}
    """
    matches = list(SECTION_PATTERN.finditer(full_text))
    if not matches:
        return {}

    seen = {}
    for m in matches:
        title = m.group(1).strip()
        prefix_match = re.match(r"(\u7b2c[\u4e00\u4e8c\u4e09\u56db\u4e94\u516d\u4e03\u516b\u4e5d\u5341\u767e]+[\u8282\u7ae0])", title)
        if prefix_match:
            key = prefix_match.group(1)
            seen[key] = (title, m.start())

    ordered = sorted(seen.values(), key=lambda x: x[1])

    sections = {}
    for i, (title, start) in enumerate(ordered):
        clean = clean_title(title)
        content_start = full_text.index("\n", start) + 1 if "\n" in full_text[start:] else start + len(title)
        if i + 1 < len(ordered):
            content_end = ordered[i + 1][1]
        else:
            content_end = len(full_text)
        content = full_text[content_start:content_end].strip()
        sections[clean] = clean_section_text(content)

    return sections


# \u6d4b\u8bd5\u5207\u5206
sections = split_sections(text)
print(f"\u5171\u5207\u51fa {len(sections)} \u4e2a\u7ae0\u8282\uff1a\n")
for title, content in sections.items():
    print(f"  {title}  ({len(content)} \u5b57)")

共切出 10 个章节：

  第一章 公司简介  (9025 字)
  第二章 会计数据和财务指标  (46345 字)
  第三章 管理层讨论与分析  (29123 字)
  第四章 公司治理  (5292 字)
  第五章 环境和社会责任  (4275 字)
  第六章 重要事项  (6451 字)
  第七章 股份变动及股东情况  (2059 字)
  第八章 优先股相关情况  (208 字)
  第九章 债券相关情况  (38 字)
  第十章 财务报告  (134986 字)


## Step 4: 保存为 JSON

In [ ]:
def process_one_pdf(pdf_filename, pdf_dir, out_dir, skip_existing=True):
    """
    \u5904\u7406\u4e00\u4efd PDF\uff1a\u63d0\u53d6\u6587\u5b57 \u2192 \u5207\u5206\u7ae0\u8282 \u2192 \u4fdd\u5b58 JSON\u3002
    \u8fd4\u56de (\u662f\u5426\u6210\u529f, \u7ae0\u8282\u6570, \u8017\u65f6\u79d2\u6570)\u3002
    """
    stock_code, company_name, year = parse_filename(pdf_filename)
    out_name = f"{stock_code}_{company_name}_{year}.json"
    out_path = os.path.join(out_dir, out_name)

    if skip_existing and os.path.exists(out_path):
        return "skipped", 0, 0

    pdf_path = os.path.join(pdf_dir, pdf_filename)
    t0 = time.time()

    try:
        full_text, total_pages = extract_text_from_pdf(pdf_path)
    except Exception as e:
        print(f"  \u63d0\u53d6\u5931\u8d25: {e}")
        return "error", 0, time.time() - t0

    sections = split_sections(full_text)

    result = {
        "meta_data": {
            "stock_code": stock_code,
            "company_name": company_name,
            "year": year,
            "total_pages": total_pages,
        },
        "sections": sections,
        "processing_info": {
            "parsed_at": datetime.now().isoformat(),
            "total_sections": len(sections),
            "total_chars": sum(len(v) for v in sections.values()),
        },
    }

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    elapsed = time.time() - t0
    return "ok", len(sections), elapsed


print("process_one_pdf \u5b9a\u4e49\u5b8c\u6210")

process_one_pdf 定义完成


## Step 5: 试跑 5 份

In [ ]:
TEST_FILES = [
    "000001_\u5e73\u5b89\u94f6\u884c_2024.pdf",
    "000009_\u4e2d\u56fd\u5b9d\u5b89_2024.pdf",
    "000002_\u4e07\u79d1A_2024.pdf",
    "000021_\u6df1\u79d1\u6280_2024.pdf",
    "600600_\u9752\u5c9b\u5564\u9152_2024.pdf",
]

for f in TEST_FILES:
    path = os.path.join(PDF_DIR, f)
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1024 / 1024 if exists else 0
    print(f"  {'OK' if exists else 'MISSING':7s} {f}  ({size_mb:.1f} MB)")

  OK      000001_平安银行_2024.pdf  (1.9 MB)
  OK      000009_中国宝安_2024.pdf  (6.4 MB)
  OK      000002_万科A_2024.pdf  (9.4 MB)
  OK      000021_深科技_2024.pdf  (1.6 MB)
  OK      600600_青岛啤酒_2024.pdf  (7.0 MB)


In [ ]:
for fname in TEST_FILES:
    print(f"\n\u5904\u7406: {fname}")
    status, n_sections, elapsed = process_one_pdf(fname, PDF_DIR, OUT_DIR, skip_existing=False)
    if status == "ok":
        print(f"  \u5b8c\u6210: {n_sections} \u4e2a\u7ae0\u8282, {elapsed:.1f}s")
    elif status == "skipped":
        print(f"  \u8df3\u8fc7\uff08\u5df2\u5b58\u5728\uff09")
    else:
        print(f"  \u5931\u8d25: {elapsed:.1f}s")

print("\n===== \u8bd5\u8dd1\u5b8c\u6210 =====")


处理: 000001_平安银行_2024.pdf
  完成: 10 个章节, 16.8s

处理: 000009_中国宝安_2024.pdf
  完成: 10 个章节, 23.4s

处理: 000002_万科A_2024.pdf
  完成: 11 个章节, 29.7s

处理: 000021_深科技_2024.pdf
  完成: 10 个章节, 13.7s

处理: 600600_青岛啤酒_2024.pdf
  完成: 10 个章节, 17.7s

===== 试跑完成 =====


In [ ]:
for fname in TEST_FILES:
    stock_code, company_name, year = parse_filename(fname)
    json_path = os.path.join(OUT_DIR, f"{stock_code}_{company_name}_{year}.json")
    if not os.path.exists(json_path):
        print(f"{fname}: \u65e0\u8f93\u51fa")
        continue
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    meta = data["meta_data"]
    info = data["processing_info"]
    print(f"\n{meta['stock_code']} {meta['company_name']} ({meta['year']})  \u2014  {meta['total_pages']} \u9875")
    print(f"  \u7ae0\u8282\u6570: {info['total_sections']}, \u603b\u5b57\u6570: {info['total_chars']}")
    for title, content in data["sections"].items():
        print(f"    {title}  ({len(content)} \u5b57)")


000001 平安银行 (2024)  —  296 页
  章节数: 10, 总字数: 237802
    第一章 公司简介  (9025 字)
    第二章 会计数据和财务指标  (46345 字)
    第三章 管理层讨论与分析  (29123 字)
    第四章 公司治理  (5292 字)
    第五章 环境和社会责任  (4275 字)
    第六章 重要事项  (6451 字)
    第七章 股份变动及股东情况  (2059 字)
    第八章 优先股相关情况  (208 字)
    第九章 债券相关情况  (38 字)
    第十章 财务报告  (134986 字)

000009 中国宝安 (2024)  —  252 页
  章节数: 10, 总字数: 269804
    第一节 重要提示、目录和释义  (1734 字)
    第二节 公司简介和主要财务指标  (3769 字)
    第三节 管理层讨论与分析  (26951 字)
    第四节 公司治理  (18161 字)
    第五节 环境和社会责任  (13740 字)
    第六节 重要事项  (4848 字)
    第七节 股份变动及股东情况  (3910 字)
    第八节 优先股相关情况  (20 字)
    第九节 债券相关情况  (2307 字)
    第十节 财务报告  (194364 字)

000002 万科A (2024)  —  327 页
  章节数: 11, 总字数: 373163
    第一节 重要提示、目录和释义  (2284 字)
    第二节 致股东  (1762 字)
    第三节 公司简介和主要财务指标  (4328 字)
    第四节 董事会报告  (78899 字)
    第五节 公司治理报告暨企业管治报告  (22440 字)
    第六节 环境和社会责任  (418 字)
    第七节 重要事项  (16018 字)
    第八节 股份变动及股东情况  (5047 字)
    第九节 监事会报告  (1825 字)
    第十节 债券相关情况  (5657 字)
    第十一节 财务报告  (234485 字)

000021 深科技 (2024)  —  218 页
  章节数:

## 批量处理所有 PDF

In [ ]:
import signal

TIMEOUT_SECONDS = 10 * 60  # 单个文件超时 10 分钟

class PdfTimeoutError(Exception):
    pass

def _timeout_handler(signum, frame):
    raise PdfTimeoutError("处理超时")

signal.signal(signal.SIGALRM, _timeout_handler)

ok_count = 0
skip_count = 0
err_count = 0
timeout_count = 0
total = len(pdf_files)

for idx, fname in enumerate(pdf_files, start=1):
    stock_code, company_name, year = parse_filename(fname)
    out_name = f"{stock_code}_{company_name}_{year}.json"
    out_path = os.path.join(OUT_DIR, out_name)

    if os.path.exists(out_path):
        skip_count += 1
        continue

    print(f"[{idx}/{total}] {fname}", end=" ... ", flush=True)

    try:
        signal.alarm(TIMEOUT_SECONDS)
        status, n_sec, elapsed = process_one_pdf(fname, PDF_DIR, OUT_DIR, skip_existing=False)
        signal.alarm(0)

        if status == "ok":
            ok_count += 1
            print(f"OK  {n_sec} 章节  {elapsed:.1f}s")
        else:
            err_count += 1
            print(f"ERROR")

    except PdfTimeoutError:
        signal.alarm(0)
        timeout_count += 1
        print(f"TIMEOUT  跳过（超过 {TIMEOUT_SECONDS // 60} 分钟）")

    except Exception as e:
        signal.alarm(0)
        err_count += 1
        print(f"ERROR  {e}")

print(f"\n===== 全部完成 =====")
print(f"成功: {ok_count}, 跳过已存在: {skip_count}, 超时: {timeout_count}, 错误: {err_count}")

[3/4790] 000004_国华网安_2024.pdf ... OK  10 章节  13.1s
[4/4790] 000006_深振业Ａ_2024.pdf ... OK  10 章节  14.4s
[5/4790] 000007_全新好_2024.pdf ... OK  10 章节  11.9s
[6/4790] 000008_神州高铁_2024.pdf ... OK  10 章节  16.0s
[8/4790] 000010_美丽生态_2024.pdf ... OK  10 章节  13.2s
[9/4790] 000011_深物业A_2024.pdf ... OK  10 章节  13.4s
[10/4790] 000012_南  玻Ａ_2024.pdf ... OK  10 章节  10.5s
[11/4790] 000014_沙河股份_2024.pdf ... OK  10 章节  8.7s
[12/4790] 000016_深康佳Ａ_2024.pdf ... OK  10 章节  15.6s
[13/4790] 000017_深中华A_2024.pdf ... OK  10 章节  13.1s
[14/4790] 000019_深粮控股_2024.pdf ... OK  10 章节  10.9s
[15/4790] 000020_深华发Ａ_2024.pdf ... OK  10 章节  13.5s
[17/4790] 000025_特  力Ａ_2024.pdf ... OK  10 章节  10.6s
[18/4790] 000027_深圳能源_2024.pdf ... OK  10 章节  28.0s
[19/4790] 000028_国药一致_2024.pdf ... OK  10 章节  17.7s
[20/4790] 000029_深深房Ａ_2024.pdf ... OK  10 章节  12.4s
[21/4790] 000030_富奥股份_2024.pdf ... OK  10 章节  18.9s
[22/4790] 000031_大悦城_2024.pdf ... OK  10 章节  26.5s
[23/4790] 000032_深桑达Ａ_2024.pdf ... OK  10 章节  21.4s
[24/4790] 000034_神州

Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMY

OK  11 章节  23.0s
[829/4790] 002353_杰瑞股份_2024.pdf ... OK  10 章节  22.4s
[830/4790] 002354_天娱数科_2024.pdf ... OK  10 章节  17.7s
[831/4790] 002355_兴民智通_2024.pdf ... OK  10 章节  13.7s
[832/4790] 002356_赫美集团_2024.pdf ... OK  10 章节  13.9s
[833/4790] 002357_富临运业_2024.pdf ... OK  10 章节  17.7s
[834/4790] 002358_森源电气_2024.pdf ... OK  10 章节  14.0s
[835/4790] 002360_同德化工_2024.pdf ... OK  10 章节  18.7s
[836/4790] 002361_神剑股份_2024.pdf ... OK  10 章节  14.5s
[837/4790] 002362_汉王科技_2024.pdf ... OK  10 章节  14.7s
[838/4790] 002363_隆基机械_2024.pdf ... OK  10 章节  11.5s
[839/4790] 002364_中恒电气_2024.pdf ... OK  10 章节  15.1s
[840/4790] 002365_永安药业_2024.pdf ... OK  10 章节  15.5s
[841/4790] 002366_融发核电_2024.pdf ... OK  10 章节  12.8s
[842/4790] 002367_康力电梯_2024.pdf ... OK  10 章节  18.0s
[843/4790] 002368_太极股份_2024.pdf ... OK  10 章节  19.7s
[844/4790] 002369_卓翼科技_2024.pdf ... OK  10 章节  16.5s
[845/4790] 002370_亚太药业_2024.pdf ... OK  10 章节  16.6s
[846/4790] 002371_北方华创_2024.pdf ... OK  10 章节  12.9s
[847/4790] 002372_伟星新材_2024.p